In [66]:
%pip -q install \
  "gymnasium==0.29.1" \
  "stable-baselines3==2.3.2" \
  "highway-env==1.8.2" \
  pygame \
  numpy>=1.24 pandas>=2.0 pyarrow>=14.0 bottleneck>=1.3.7 Pillow>=9.5 \
  scikit-learn>=1.3 \
  sentence-transformers>=2.5.0 \
  open-clip-torch>=2.24.0 \
  transformers>=4.41.0 \
  accelerate>=0.30.0 \
  safetensors \
  tensorboard \
  pathlib

zsh:1: 1.24 not found
Note: you may need to restart the kernel to use updated packages.


In [88]:
import sys, inspect
import rl_langvision, rl_langvision.reward_wrappers as rw
print("pkg:", rl_langvision.__file__)
print("reward_wrappers:", rw.__file__)
print(inspect.getsource(rw.SafetySpeedRewardWrapper)[:300])  # should show your _base() unwrapping


pkg: /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/rl_langvision/__init__.py
reward_wrappers: /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/rl_langvision/reward_wrappers.py
class SafetySpeedRewardWrapper(gym.Wrapper):
    """
    Shaped reward for the ambulance (ego).

      total_r = + speed_w * norm_speed
                + clear_bonus (if no blockers ahead)
                - block_penalty_w * (#blockers ahead)
                - ttc_w * penalty when TTC < threshold
  


In [90]:
import importlib
import rl_langvision.reward_wrappers as rw
rw = importlib.reload(rw)  # now rw.SafetySpeedRewardWrapper is the new class
from rl_langvision.reward_wrappers import SafetySpeedRewardWrapper  # re-import symbol
import os


In [91]:
# 🚫 Quiet the noisy warnings/logs (run this first)
import os, warnings

# Silence Gymnasium UserWarnings
warnings.filterwarnings("ignore", category=UserWarning, module="gymnasium")
warnings.filterwarnings("ignore", message=r".*env\.vehicle.*", category=UserWarning)
warnings.filterwarnings("ignore", message=r".*env\.road.*", category=UserWarning)

# Optional: hide extra backend logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gymnasium as gym
try:
    gym.logger.set_level(40)  # ERROR
except Exception:
    pass


In [92]:
from __future__ import annotations
import os
from pathlib import Path
import numpy as np
import torch
import gymnasium as gym
import highway_env  # registers envs
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy




In [93]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"   # headless pygame (no window)

import gymnasium as gym
import highway_env          # <-- this import registers the env IDs

# List what's registered
available = [spec.id for spec in gym.envs.registry.values() if "highway" in spec.id.lower()]
print("Highway envs registered:", available)

# This should succeed:
gym.spec("highway-v0")
print("✅ 'highway-v0' is available.")


Highway envs registered: ['highway-v0', 'highway-fast-v0']
✅ 'highway-v0' is available.


In [94]:

import os, sys, random, argparse, json
from pathlib import Path
from typing import Iterable, Dict, List, Optional


from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import (
    DummyVecEnv, SubprocVecEnv, VecMonitor, VecNormalize
)
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, BaseCallback



In [95]:
import os, sys
from pathlib import Path

# If this runs from a file inside tools/, infer the repo root; otherwise fall back.
def _guess_root():
    # __file__ exists in scripts; in notebooks it doesn't
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    # if we're in .../Ambulance_EGO_4500/tools, root is parent of 'tools'
    if here.name == "tools" and (here.parent / "rl_langvision").exists():
        return here.parent
    # custom absolute fallback (edit if needed)
    return Path("/Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500").resolve()

PROJ_ROOT = _guess_root()                      # e.g., .../Ambulance_EGO_4500
PKG_DIR   = PROJ_ROOT / "rl_langvision"        # Path, not str

assert PROJ_ROOT.exists(), f"Project root not found: {PROJ_ROOT}"
PKG_DIR.mkdir(parents=True, exist_ok=True)

# Make it a package so 'import rl_langvision.*' works
init_file = PKG_DIR / "__init__.py"
init_file.touch(exist_ok=True)

# Put the project root (parent of package) on sys.path
root_str = str(PROJ_ROOT)
if root_str not in sys.path:
    sys.path.insert(0, root_str)

print("Using PROJ_ROOT =", PROJ_ROOT)
print("Package dir     =", PKG_DIR)
print("sys.path[0]     =", sys.path[0])

# quick smoke test
try:
    import rl_langvision
    print("Imported rl_langvision OK from:", rl_langvision.__file__)
except Exception as e:
    print("Import failed:", e)


Using PROJ_ROOT = /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500
Package dir     = /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/rl_langvision
sys.path[0]     = /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500
Imported rl_langvision OK from: /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/rl_langvision/__init__.py


In [72]:
import pkgutil, importlib
mods = [m.name for m in pkgutil.iter_modules([str(PKG_DIR)])]
print('rl_langvision modules:', mods)

# Example import(s) — adjust to the exact names you have
# from rl_langvision.features_extractor_clip import YourFeatureExtractorClass
# from rl_langvision.amb_highway_wrapper_clip import YourWrapperClass


rl_langvision modules: ['amb_highway_wrapper_clip', 'cached_embedder', 'clip_embedder', 'features_extractor_clip', 'language_embedder', 'reward_wrappers', 'telemetry_wrapper', 'yielding_traffic']


In [96]:
import torch, sys
print("Py:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("MPS built:", hasattr(torch.backends, "mps") and torch.backends.mps.is_built())

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)


Py: 3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ]
CUDA available: False
MPS available: True
MPS built: True
Using device: mps


In [97]:
!nvidia-smi
import torch, sys
print("Py:", sys.version)
print("CUDA available:", torch.cuda.is_available())


zsh:1: command not found: nvidia-smi
Py: 3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ]
CUDA available: False


In [98]:
# project modules
from rl_langvision.clip_embedder import CLIPImageEncoder
from rl_langvision.cached_embedder import CachedLLMEmbedder
from rl_langvision.language_embedder import FrozenTextEmbedder
from rl_langvision.amb_highway_wrapper_clip import AmbulanceHighwayCLIPWrapper
from rl_langvision.features_extractor_clip import CLIPLangExtractor
from rl_langvision.reward_wrappers import SafetySpeedRewardWrapper

# registers "rl_langvision.yielding_traffic.YieldingIDM"
import rl_langvision.yielding_traffic  # noqa: F401



In [99]:
# rl_langvision/reward_wrappers.py
from __future__ import annotations
import numpy as np
import gymnasium as gym

def _unit(x: float, lo: float, hi: float) -> float:
    if hi <= lo:
        return 0.0
    return float(np.clip((x - lo) / (hi - lo), 0.0, 1.0))

class SafetySpeedRewardWrapper(gym.Wrapper):
    """
    Shaped reward for the ambulance (ego).

      total_r = + speed_w * norm_speed
                + clear_bonus (if no blockers ahead)
                - block_penalty_w * (#blockers ahead)
                - ttc_w * penalty when TTC < threshold
                - collision_penalty (only on crash)

    Notes
    -----
    - Always reads ego/road from the *base* HighwayEnv (self._base()) so it
      still works when other wrappers are stacked above.
    """

    def __init__(self, env: gym.Env, cfg: dict | None = None):
        super().__init__(env)
        self.cfg = {
            "speed_min": 20.0,    # m/s
            "speed_max": 30.0,    # m/s
            "ahead_dist": 35.0,   # m
            "same_lane_only": True,
            "ttc_threshold": 2.0, # s
            "ttc_w": 0.30,
            "speed_w": 1.00,
            "clear_bonus": 0.30,
            "block_penalty_w": 0.15,
            "collision_penalty": 1.00,
            "clip_abs": 1.0,
        }
        if cfg:
            self.cfg.update(cfg)

    # ---------- helpers over base env ----------
    def _base(self):
        """Return the unwrapped base env (HighwayEnv)."""
        env = self.env
        # unwrap through any number of wrappers
        while hasattr(env, "env"):
            env = env.env
        return env

    def _ego(self):
        base = self._base()
        return getattr(base, "vehicle", None) or getattr(base, "ego_vehicle", None)

    def _road(self):
        return getattr(self._base(), "road", None)

    def _same_lane(self, a, b) -> bool:
        try:
            return a.lane_index[2] == b.lane_index[2]
        except Exception:
            return True

    def _blockers_ahead(self, ego) -> int:
        road = self._road()
        if road is None:
            return 0
        ahead = 0
        for v in getattr(road, "vehicles", ()):
            if v is ego:
                continue
            dx = v.position[0] - ego.position[0]
            if dx <= 0.0 or dx > self.cfg["ahead_dist"]:
                continue
            if self.cfg["same_lane_only"] and not self._same_lane(v, ego):
                continue
            ahead += 1
        return ahead

    def _min_ttc(self, ego) -> float:
        road = self._road()
        if road is None:
            return 1e8
        min_ttc = 1e9
        for v in getattr(road, "vehicles", ()):
            if v is ego:
                continue
            dx = v.position[0] - ego.position[0]
            if dx <= 0.0 or dx > self.cfg["ahead_dist"]:
                continue
            if self.cfg["same_lane_only"] and not self._same_lane(v, ego):
                continue
            rel_v = (ego.speed - v.speed) + 1e-6
            if rel_v <= 0:
                continue
            ttc = dx / rel_v
            if ttc < min_ttc:
                min_ttc = ttc
        return min_ttc if min_ttc < 1e8 else 1e8

    # ---------- main ----------
    def step(self, action):
        obs, base_r, terminated, truncated, info = self.env.step(action)
        info = dict(info) if info is not None else {}

        ego = self._ego()
        if ego is None:
            # keep keys for TB even if something goes wrong
            info.update({
                "r_speed": 0.0, "r_clear": 0.0, "r_block": 0.0,
                "r_ttc_pen": 0.0, "r_collision": 0.0,
                "blockers_ahead": 0, "ttc_min": float(1e8),
                "speed_kph": 0.0, "crashed": bool(info.get("crashed", False)),
            })
            return obs, base_r, terminated, truncated, info

        # speed
        speed_mps = float(getattr(ego, "speed", 0.0))
        speed_kph = speed_mps * 3.6
        vnorm = _unit(speed_mps, self.cfg["speed_min"], self.cfg["speed_max"])
        r_speed = self.cfg["speed_w"] * vnorm

        # corridor & blockers
        blockers = self._blockers_ahead(ego)
        r_clear = self.cfg["clear_bonus"] if blockers == 0 else 0.0
        r_block = - self.cfg["block_penalty_w"] * float(blockers)

        # TTC penalty
        ttc = self._min_ttc(ego)
        r_ttc_pen = 0.0
        if ttc < self.cfg["ttc_threshold"]:
            r_ttc_pen = self.cfg["ttc_w"] * (1.0 - (ttc / max(self.cfg["ttc_threshold"], 1e-6)))

        # crash
        crashed = bool(info.get("crashed", getattr(ego, "crashed", False)))
        r_collision = - self.cfg["collision_penalty"] if crashed else 0.0

        # total shaped reward (replace env reward)
        shaped = r_speed + r_clear + r_block - r_ttc_pen + r_collision
        total_r = shaped
        ca = self.cfg.get("clip_abs", None)
        if ca is not None:
            total_r = float(np.clip(total_r, -float(ca), float(ca)))

        # log
        info.update({
            "r_speed": float(r_speed),
            "r_clear": float(r_clear),
            "r_block": float(r_block),
            "r_ttc_pen": float(r_ttc_pen),
            "r_collision": float(r_collision),
            "blockers_ahead": int(blockers),
            "ttc_min": float(ttc),
            "speed_kph": float(speed_kph),
            "crashed": crashed,
        })

        return obs, total_r, terminated, truncated, info


# Training version 2
model folder save to Ego_Emergency


In [ ]:
# tools/train_ppo_clip_vlm_vecnorm.py
from __future__ import annotations
import os, sys, json, random, warnings, logging, glob, time, pickle, difflib
from pathlib import Path
from typing import Dict, Iterable, Optional, List

# ---------------- Silence noisy logs ----------------
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("stable_baselines3").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# ---------------- Core ----------------
import numpy as np
import torch
import gymnasium as gym
import highway_env
import pandas as pd

# ---------------- Project paths ----------------
PROJ_ROOT = Path("/Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500").resolve()
assert PROJ_ROOT.exists(), f"Project root not found: {PROJ_ROOT}"
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

# ---------------- Project modules ----------------
from rl_langvision.clip_embedder import CLIPImageEncoder
from rl_langvision.cached_embedder import CachedLLMEmbedder
from rl_langvision.amb_highway_wrapper_clip import AmbulanceHighwayCLIPWrapper
from rl_langvision.reward_wrappers import SafetySpeedRewardWrapper
from rl_langvision.yielding_traffic import YieldingIDM  # noqa: F401
from rl_langvision.features_extractor_clip import CLIPLangExtractor
from rl_langvision.language_embedder import FrozenTextEmbedder

# ---------------- SB3 ----------------
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback

# ---------------- Cached text embeddings dir ----------------
DATA_ROOT = PROJ_ROOT / "ambulance_dataset_fast_150_espisode_cpu_30_senario"
CACHED_DIR = str(DATA_ROOT / "cached_llm_v2")
assert os.path.isdir(CACHED_DIR), f"Missing cache dir: {CACHED_DIR}"

# ---------------- Device & seeds ----------------
def _pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        return "mps"
    return "cpu"

def set_global_seeds(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print("Using device:", _pick_device())

# ---------------- Load scenarios from manifests ----------------
MANIFEST_DIR = DATA_ROOT / "manifests"
MANIFESTS = {
    "train": MANIFEST_DIR / "episodes_train.jsonl",
    "val":   MANIFEST_DIR / "episodes_val.jsonl",
    "test":  MANIFEST_DIR / "episodes_test.jsonl",
}

def _read_unique_scenarios(jsonl_path: Path) -> list[str]:
    uniq, seen = [], set()
    if not jsonl_path.exists():
        return uniq
    with open(jsonl_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            scen = str(rec.get("scenario", "")).strip()
            if scen and scen not in seen:
                seen.add(scen); uniq.append(scen)
    return uniq

def fill_config_scenarios_from_manifests(cfg_env: dict) -> None:
    for split, path in MANIFESTS.items():
        names = _read_unique_scenarios(path)
        key = {"train":"scenarios_train","val":"scenarios_val","test":"scenarios_test"}[split]
        if names:
            cfg_env[key] = names

# ---------------- Config ----------------
CONFIG = {
    "log_dir": str(PROJ_ROOT / "runs" / "Ego_Emergency"),
    "output_dir": str(PROJ_ROOT),
    "reward": {
        "mode": "custom",
        "custom_cfg": {
            "speed_min": 20.0, "speed_max": 35.0,
            "ahead_dist": 40.0, "same_lane_only": True,
            "ttc_threshold": 3.2, "ttc_w": 1.00,
            "speed_w": 0.90, "clear_bonus": 0.30,
            "block_penalty_w": 0.10, "collision_penalty": 3.0,
            "clip_abs": 1.0,
        }
    },
    "env": {
        "lanes_count": 4, "vehicles_count": 45, "duration": 50,
        "simulation_frequency": 15, "policy_frequency": 1,
        "offscreen_rendering": True, "render_agent": True,
        "show_trajectories": False,
        "high_speed_reward": 0.0, "right_lane_reward": 0.0, "lane_change_reward": 0.0,
        "reward_speed_range": [22, 32], "normalize_reward": False,
        "centering_position": [0.3, 0.5], "scaling": 5.5,
        # scenario lists will be overwritten by manifests (below)
        "scenarios_train": ["highway_emergency_dense", "highway_time_pressure", "highway_merge_heavy", "highway_rush_hour"],
        "scenarios_val":   ["highway_emergency_moderate", "highway_stop_and_go"],
        "scenarios_test":  ["highway_construction", "highway_lane_closure", "highway_accident_scene"],
        "is_emergency": True, "yield_radius": 60.0, "yield_right_bias": 0.7,
        "other_vehicles_type": "rl_langvision.yielding_traffic.YieldingIDM",
    },
    "vision": {"clip_model": "openai/clip-vit-base-patch32", "device": None},
    "text": {"cached_llm_dir": CACHED_DIR, "cached_dim": 384, "local_model": None},
    "policy": {"feat_dim": 512},
    "wrapper": {"clip_stride": 4},
    "algo": {
        "n_envs": 2,
        "ppo": {
            "n_steps": 2048, "batch_size": 512, "n_epochs": 15, "lr": 5e-5,
            "gamma": 0.99, "clip_range": 0.2, "gae_lambda": 0.95,
            "ent_coef": 0.005, "target_kl": 0.04, "vf_coef": 0.5, "max_grad_norm": 0.5,
        },
    },
}
# overwrite scenario lists from manifests if available
fill_config_scenarios_from_manifests(CONFIG["env"])

# ---------------- Logging callbacks ----------------
class InfoKeysLogger(BaseCallback):
    def __init__(self, keys=("blockers_ahead","ttc_min","r_speed","r_clear","r_block","r_ttc_pen","r_collision","speed_kph"), verbose=0):
        super().__init__(verbose); self.keys = keys
    def _on_step(self) -> bool:
        infos = self.locals.get("infos") or []
        agg: Dict[str, list] = {k: [] for k in self.keys}
        for d in infos:
            for k in self.keys:
                if k in d: agg[k].append(d[k])
        for k, vals in agg.items():
            if vals:
                try: self.logger.record_mean(f"info/{k}", float(np.mean(vals)))
                except Exception: self.logger.record(f"info/{k}", float(np.mean(vals)))
        return True

class EpisodeRewardComponentsLogger(BaseCallback):
    def __init__(self, keys=("r_speed","r_ttc_pen","r_clear","r_block","r_collision"), verbose=0):
        super().__init__(verbose); self.keys = keys
    def _on_training_start(self) -> None:
        n_envs = self.training_env.num_envs
        self.buf = {k: np.zeros(n_envs, np.float32) for k in self.keys}
        self.crash = np.zeros(n_envs, np.int32)
    def _on_step(self) -> bool:
        infos = self.locals.get("infos") or []
        dones = self.locals.get("dones"); dones_seq = np.asarray(dones).ravel() if dones is not None else ()
        for i, inf in enumerate(infos):
            for k in self.keys:
                if k in inf: self.buf[k][i] += float(inf[k])
            if inf.get("crashed", False) or float(inf.get("r_collision", 0.0)) < 0.0:
                self.crash[i] = 1
            if i < len(dones_seq) and bool(dones_seq[i]):
                for k in self.keys:
                    self.logger.record(f"components/ep_{k}", float(self.buf[k][i])); self.buf[k][i] = 0.0
                self.logger.record("components/ep_collision", float(self.crash[i])); self.crash[i] = 0
        return True

class ConsolePrinter(BaseCallback):
    def __init__(self, max_lines_per_rollout: int = 40, verbose: int = 0):
        super().__init__(verbose); self.max_lines_per_rollout = max_lines_per_rollout
    def _on_training_start(self) -> None:
        n = self.training_env.num_envs
        self.sum_r = np.zeros(n, np.float64); self.len_steps = np.zeros(n, np.int32)
        self.comp_keys = ("r_speed","r_ttc_pen","r_clear","r_block","r_collision")
        self.comp_sum = {k: np.zeros(n, np.float64) for k in self.comp_keys}
        self.blockers_pk = np.zeros(n, np.float64)
        self.speed_sum = np.zeros(n, np.float64); self.speed_cnt = np.zeros(n, np.int32)
        self.crashed = np.zeros(n, np.int32)
        self.finished = []; self.lines_this_rollout = 0; self.rollout_idx = 0
        n_steps = int(self.model.n_steps); rollout = n_steps * n
        device = getattr(self.model, "device", "cpu")
        print(f"=== TRAIN START === device={device} n_envs={n} n_steps={n_steps} rollout_size={rollout}")
    def _on_step(self) -> bool:
        rewards = self.locals.get("rewards"); dones = self.locals.get("dones"); infos = self.locals.get("infos") or []
        if rewards is None: return True
        dones_seq = np.asarray(dones).ravel() if dones is not None else ()
        for i, inf in enumerate(infos):
            self.sum_r[i] += float(rewards[i]); self.len_steps[i] += 1
            for k in self.comp_keys:
                if k in inf: self.comp_sum[k][i] += float(inf[k])
            if "speed_kph" in inf:
                self.speed_sum[i] += float(inf["speed_kph"]); self.speed_cnt[i] += 1
            if "blockers_ahead" in inf:
                self.blockers_pk[i] = max(self.blockers_pk[i], float(inf["blockers_ahead"]))
            if inf.get("crashed", False) or float(inf.get("r_collision", 0.0)) < 0.0:
                self.crashed[i] = 1
            if i < len(dones_seq) and bool(dones_seq[i]):
                spd = (self.speed_sum[i] / max(1, self.speed_cnt[i]))
                row = {"R": self.sum_r[i], "L": int(self.len_steps[i]), "spd": spd,
                       "crash": int(self.crashed[i]),
                       **{k: float(self.comp_sum[k][i]) for k in self.comp_keys},
                       "blk_max": int(self.blockers_pk[i])}
                self.finished.append(row)
                if self.lines_this_rollout < self.max_lines_per_rollout:
                    print(f"[EP] R={row['R']:+.3f}  len={row['L']:4d}  speed={row['spd']:5.1f}kph  "
                          f"blk_max={row['blk_max']:2d}  crash={bool(row['crash'])}  "
                          f"comp:(speed={row['r_speed']:+.3f}, clear={row['r_clear']:+.3f}, "
                          f"block={row['r_block']:+.3f}, ttc_pen=-{row['r_ttc_pen']:.3f}, "
                          f"coll={row['r_collision']:+.3f})")
                    self.lines_this_rollout += 1
                # reset per-env
                self.sum_r[i] = 0.0; self.len_steps[i] = 0
                self.blockers_pk[i] = 0.0; self.crashed[i] = 0
                self.speed_sum[i] = 0.0; self.speed_cnt[i] = 0
                for k in self.comp_keys: self.comp_sum[k][i] = 0.0
        return True
    def _on_rollout_end(self) -> bool:
        self.rollout_idx += 1
        if self.finished:
            R = np.array([e["R"] for e in self.finished], np.float64)
            L = np.array([e["L"] for e in self.finished], np.float64)
            C = np.array([e["crash"] for e in self.finished], np.float64)
            S = np.array([e["spd"] for e in self.finished], np.float64)
            print(f"[ROLLOUT {self.rollout_idx}] episodes={len(self.finished)}  "
                  f"Rmean={R.mean():+.3f}  Lmean={L.mean():.1f}  "
                  f"crash_rate={C.mean():.2f}  speed_mean={S.mean():.1f}kph")
            self.finished.clear()
        self.lines_this_rollout = 0
        return True

# ---------- Best model mirroring (safe) ----------
class BestSaver(BaseCallback):
    """
    Mirror EvalCallback's best zip into:
      - best_model.pkl: model.get_parameters() only (picklable)
      - vecnorm.pkl: current VecNormalize stats
    """
    def __init__(self, train_env: VecNormalize, run_dir: Path, verbose=0):
        super().__init__(verbose)
        self.train_env = train_env
        self.run_dir = Path(run_dir)
        self.best_zip = self.run_dir / "best" / "best_model.zip"   # written by EvalCallback
        self.best_pkl = self.run_dir / "best" / "best_model.pkl"
        self.vecnorm_pkl = self.run_dir / "vecnorm.pkl"
        self._last_best_mtime = None

    def _on_step(self) -> bool:
        if self.best_zip.exists():
            mtime = self.best_zip.stat().st_mtime
            if mtime != self._last_best_mtime:
                self._last_best_mtime = mtime
                params = self.model.get_parameters()
                with open(self.best_pkl, "wb") as f:
                    pickle.dump(params, f)
                try:
                    self.train_env.save(str(self.vecnorm_pkl))
                except Exception as e:
                    print(f"[BestSaver] VecNormalize save failed: {e}")
                print(f"💾 Updated: best_model.pkl (params) and vecnorm.pkl")
        return True

# ---------------- Scenario / encoder helpers ----------------
def _choose_scenario(cfg_env: dict, mode: str) -> dict:
    cfg = dict(cfg_env)
    key = {"train":"scenarios_train","eval":"scenarios_val","test":"scenarios_test"}[mode]
    scenarios: Optional[List[str]] = cfg.get(key)
    if scenarios: cfg["scenario"] = random.choice(scenarios)
    return cfg

def _map_clip_name(name: str):
    low = (name or "").lower()
    # map "openai/clip-vit-base-patch32" -> ("ViT-B-32", "openai") for open_clip
    if "vit-base-patch32" in low: return "ViT-B-32", "openai"
    if name.lower() in {"vit-b-32","vit-b-32-quickgelu"}: return name, "openai"
    return name, "openai"

def _assert_cached_has_scenario(scenario: str):
    """Optional fast check: scenario label visible in cached parquet (first file)."""
    try:
        parqs = sorted(Path(CACHED_DIR).glob("*.parquet"))
        if not parqs: return
        df = pd.read_parquet(parqs[0])
        if "scenario" not in df.columns: return
        names = set(str(s) for s in df["scenario"].dropna().unique())
        if scenario and scenario not in names:
            hint = difflib.get_close_matches(scenario, list(names), n=5, cutoff=0.3)
            raise RuntimeError(f"Scenario '{scenario}' not found in cached text. Closest: {hint}")
    except Exception:
        # don't hard-stop if parquet schema differs
        pass

def _make_env_thunk(config: dict, seed: int, mode: str):
    assert mode in {"train","eval","test"}
    def _init():
        cfg = _choose_scenario(config.get("env", {}) or {}, mode)
        base_env = gym.make("highway-v0", render_mode="rgb_array", config=cfg)
        try: base_env.reset(seed=seed)
        except TypeError: pass
        try:
            base_env.action_space.seed(seed)
            base_env.observation_space.seed(seed)
        except Exception: pass

        # Encoders
        vcfg = config.get("vision", {}) or {}
        clip_dev = vcfg.get("device") or _pick_device()
        model_name, pretrained = _map_clip_name(vcfg.get("clip_model", "openai/clip-vit-base-patch32"))
        clip_enc = CLIPImageEncoder(model_name=model_name, pretrained=pretrained, device=clip_dev)

        tcfg = config.get("text", {}) or {}
        if tcfg.get("cached_llm_dir"):
            cached_llm = CachedLLMEmbedder(tcfg["cached_llm_dir"], dim=int(tcfg.get("cached_dim", 384)))
            text_emb = None
            _assert_cached_has_scenario(cfg.get("scenario", ""))
        else:
            text_emb = FrozenTextEmbedder(tcfg.get("local_model", "sentence-transformers/all-MiniLM-L6-v2"))
            cached_llm = None

        env = AmbulanceHighwayCLIPWrapper(
            base_env, clip_enc,
            text_embedder=text_emb, cached_llm=cached_llm,
            clip_stride=int(config.get("wrapper", {}).get("clip_stride", 4)),
        )

        # Reward wrapper last
        if (config.get("reward") or {}).get("mode", "custom") == "custom":
            env = SafetySpeedRewardWrapper(env, cfg=config["reward"]["custom_cfg"])

        # Assert wrapper emits dict + small debug print
        o, _ = env.reset()
        assert isinstance(o, dict) and "clip" in o and "text" in o, "Obs must be Dict({'clip','text'})"
        txt = o.get("text", None)
        if isinstance(txt, np.ndarray) and txt.ndim == 1:
            print(f"[ENV] scenario={cfg.get('scenario','?')} | text-emb-dim={txt.shape[0]}")
        else:
            print(f"[ENV] scenario={cfg.get('scenario','?')} | text key type={type(txt)}")
        return env
    return _init

def make_vec_envs(config: dict, n_envs: int, base_seed: int, mode: str):
    thunks = [_make_env_thunk(config, base_seed + i, mode) for i in range(max(1, n_envs))]
    venv = DummyVecEnv(thunks)
    venv = VecMonitor(venv)
    venv = VecNormalize(venv, norm_obs=True, norm_reward=False, training=(mode=="train"))
    o = venv.reset()
    assert isinstance(o, dict) and "clip" in o and "text" in o, "Vec env must emit Dict({'clip','text'})"
    return venv

def _sync_vecnorm_stats(train_env: VecNormalize, other_env: VecNormalize):
    other_env.obs_rms = train_env.obs_rms
    other_env.ret_rms = train_env.ret_rms
    other_env.training = False
    other_env.norm_reward = False

# ---------------- Evaluation ----------------
def _bool_flag(x):
    if isinstance(x, (list, tuple, np.ndarray)): return bool(np.asarray(x).any())
    return bool(x)

def evaluate_model(model: PPO, env: VecNormalize, episodes: int = 10, deterministic: bool = True) -> Dict[str, float]:
    returns, lengths, speed_kph_all = [], [], []
    collisions = 0
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False; truncated = False
        tot = 0.0; steps = 0
        while not (_bool_flag(done) or _bool_flag(truncated)):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, done, truncated, infos = env.step(action)
            tot += float(np.asarray(r).mean()); steps += 1
            info0 = (infos[0] if isinstance(infos, (list, tuple)) and infos else infos) or {}
            if info0.get("crashed", False): collisions += 1
            if "speed_kph" in info0: speed_kph_all.append(float(info0["speed_kph"]))
        returns.append(tot); lengths.append(steps)
    return {
        "return_mean": float(np.mean(returns)) if returns else 0.0,
        "return_std":  float(np.std(returns)) if returns else 0.0,
        "len_mean":    float(np.mean(lengths)) if lengths else 0.0,
        "collisions":  int(collisions),
        "speed_kph_mean": float(np.mean(speed_kph_all)) if speed_kph_all else 0.0,
    }

# ---------------- Training driver ----------------
def print_seed_header(cfg: dict, seed: int, total_timesteps: int):
    a = (cfg.get("algo") or {}).get("ppo", {})
    n_envs = int((cfg.get("algo") or {}).get("n_envs", 1))
    n_steps = int(a.get("n_steps", 2048))
    rollout = n_envs * n_steps
    print("----- RUN CONFIG -----\n"
          f"seed={seed}\n"
          f"timesteps={total_timesteps:,}\n"
          f"n_envs={n_envs}, n_steps={n_steps}, rollout_size={rollout}\n"
          f"lr={a.get('lr')}, batch_size={a.get('batch_size')}, gamma={a.get('gamma')}, "
          f"clip={a.get('clip_range')}, ent_coef={a.get('ent_coef')}, target_kl={a.get('target_kl')}\n"
          f"reward_mode={(cfg.get('reward') or {}).get('mode', 'custom')}\n"
          f"train_scenarios={cfg['env'].get('scenarios_train')[:5]}…({len(cfg['env'].get('scenarios_train',[]))})\n"
          f"val_scenarios={cfg['env'].get('scenarios_val')[:5]}…({len(cfg['env'].get('scenarios_val',[]))})\n"
          f"test_scenarios={cfg['env'].get('scenarios_test')[:5]}…({len(cfg['env'].get('scenarios_test',[]))})\n"
          "----------------------")

def train_one_seed(config: dict, seed: int, total_timesteps: int, model_prefix: str):
    set_global_seeds(seed)
    n_envs = int((config.get("algo") or {}).get("n_envs", 1))

    tb_root = Path(config.get("log_dir") or (PROJ_ROOT / "runs" / "ppo_simple"))
    run_dir = tb_root / f"seed_{seed}"
    ckpt_dir = run_dir / "ckpt"
    best_dir = run_dir / "best"
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_dir.mkdir(parents=True, exist_ok=True)

    # build envs
    env = make_vec_envs(config, n_envs, base_seed=seed, mode="train")
    eval_env = make_vec_envs(config, 1, base_seed=seed + 10_000, mode="eval")
    _sync_vecnorm_stats(env, eval_env)

    # policy
    policy_kwargs = dict(
        features_extractor_class=CLIPLangExtractor,
        features_extractor_kwargs=dict(features_dim=(config.get("policy") or {}).get("feat_dim", 512)),
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    )
    a = (config.get("algo") or {}).get("ppo", {}) or {}
    model = PPO(
        "MultiInputPolicy", env, seed=seed,
        policy_kwargs=policy_kwargs,
        n_steps=a.get("n_steps", 2048),
        batch_size=a.get("batch_size", 512),
        n_epochs=a.get("n_epochs", 15),
        learning_rate=a.get("lr", 5e-5),
        gamma=a.get("gamma", 0.99),
        clip_range=a.get("clip_range", 0.2),
        gae_lambda=a.get("gae_lambda", 0.95),
        ent_coef=a.get("ent_coef", 0.005),
        target_kl=a.get("target_kl", 0.04),
        verbose=1, tensorboard_log=str(tb_root), device=_pick_device(),
    )

    # callbacks
    console_cb = ConsolePrinter(max_lines_per_rollout=40)
    info_cb = InfoKeysLogger()
    comp_cb = EpisodeRewardComponentsLogger()
    eval_cb = EvalCallback(
        eval_env, best_model_save_path=str(best_dir),
        log_path=str(run_dir / "eval"),
        eval_freq=max(1, (10_000 // max(1, n_envs))),
        n_eval_episodes=20, deterministic=True, render=False,
    )
    best_mirror_cb = BestSaver(train_env=env, run_dir=run_dir)
    ckpt_cb = CheckpointCallback(
        save_freq=max(1, (50_000 // max(1, n_envs))),
        save_path=str(ckpt_dir), name_prefix="ppo"
    )

    print_seed_header(config, seed, total_timesteps)
    print(f"[seed {seed}] learn() for {total_timesteps:,} steps on {model.device} …")
    t0 = time.time()
    model.learn(
        total_timesteps=total_timesteps,
        tb_log_name=f"seed_{seed}",
        callback=[console_cb, info_cb, comp_cb, eval_cb, best_mirror_cb, ckpt_cb],
    )
    print(f"[seed {seed}] learn() done in {(time.time()-t0)/60:.1f} min")

    # final saves
    model_path = Path(config.get("output_dir", PROJ_ROOT)) / f"{model_prefix}_seed{seed}.zip"
    model.save(str(model_path))
    env.save(str(run_dir / "vecnorm.pkl"))  # final vecnorm snapshot
    print(f"[seed {seed}] saved -> {model_path}")
    print(f"[seed {seed}] best model dir -> {best_dir}")
    print(f"[seed {seed}] vecnorm -> {run_dir / 'vecnorm.pkl'}")

    # quick metrics
    val_metrics = evaluate_model(model, eval_env, episodes=20, deterministic=True)
    (run_dir / "val_metrics.json").write_text(json.dumps(val_metrics, indent=2))
    print(f"[seed {seed}] VAL:", val_metrics)

    test_env = make_vec_envs(config, 1, base_seed=seed + 20_000, mode="test")
    _sync_vecnorm_stats(env, test_env)
    test_metrics = evaluate_model(model, test_env, episodes=20, deterministic=True)
    (run_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
    print(f"[seed {seed}] TEST:", test_metrics)

    env.close(); eval_env.close(); test_env.close()

def train(config: dict, seeds: Iterable[int] | None = None,
          total_timesteps: int = 100_000, model_prefix: str = "ppo_clip_vlm_f2"):
    seeds = list(seeds) if seeds is not None else [0]
    for s in seeds:
        train_one_seed(config, seed=s, total_timesteps=total_timesteps, model_prefix=model_prefix)

# ---------------- Main ----------------
if __name__ == "__main__":
    # 1) cache sanity
    embed_info = os.path.join(CACHED_DIR, "embed_info.json")
    assert os.path.exists(embed_info)
    _info = json.loads(open(embed_info).read())
    assert int(_info["dim"]) == 384
    parqs = glob.glob(os.path.join(CACHED_DIR, "*.parquet")); assert parqs
    _df = pd.read_parquet(parqs[0])
    for col in ["episode_id","step","embedding","text"]:
        assert col in _df.columns
    assert len(_df["embedding"].iloc[0]) == 384
    print("[ok] cache sanity:", Path(parqs[0]).name, "rows=", len(_df))

    # 2) smoke env
    _env = make_vec_envs(CONFIG, 1, base_seed=0, mode="train")
    obs = _env.reset()
    assert isinstance(obs, dict) and all(k in obs for k in ("clip","text")), "Obs must have 'clip' and 'text'"
    print("Obs keys:", list(obs.keys()))
    _env.close()

    # 3) train
    set_global_seeds(0)
    train(CONFIG, seeds=[0, 1], total_timesteps=600_000, model_prefix="ppo_clip_vlm_f2")


Using device: mps
[ok] cache sanity: 20251007_011932-88adade2.parquet rows= 47
[ENV] scenario=roundabout_multi_lane | text-emb-dim=384
Obs keys: ['clip', 'text']
[ENV] scenario=roundabout_multi_lane | text-emb-dim=384
[ENV] scenario=roundabout_congested | text-emb-dim=384
[ENV] scenario=highway_emergency_moderate | text-emb-dim=384
Using mps device
----- RUN CONFIG -----
seed=0
timesteps=1,000,000
n_envs=2, n_steps=2048, rollout_size=4096
lr=5e-05, batch_size=512, gamma=0.99, clip=0.2, ent_coef=0.005, target_kl=0.04
reward_mode=custom
train_scenarios=['transition_highway_urban', 'night_emergency_response', 'merge_heavy_traffic', 'merge_zipper_pattern', 'merge_multi_point']…(22)
val_scenarios=['highway_emergency_moderate', 'highway_stop_and_go']…(2)
test_scenarios=['transition_highway_urban', 'night_emergency_response']…(2)
----------------------
[seed 0] learn() for 1,000,000 steps on mps …
Logging to /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/runs/Ego_

KeyboardInterrupt: 

# Old version

In [ ]:
# tools/train_ppo_clip_vlm_vecnorm.py
from __future__ import annotations
import os, sys, json, random, warnings, logging, glob, time, pickle
from pathlib import Path
from typing import Dict, Iterable, Optional, List
from rl_langvision.language_embedder import FrozenTextEmbedder


# ---------------- Silence noisy logs ----------------
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("stable_baselines3").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# ---------------- Core ----------------
import numpy as np
import torch
import gymnasium as gym
import highway_env
import pandas as pd

# ---------------- Project paths ----------------
PROJ_ROOT = Path("/Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500").resolve()
assert PROJ_ROOT.exists(), f"Project root not found: {PROJ_ROOT}"
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))



# ---------------- SB3 ----------------
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback

# ---------------- Cached text embeddings dir ----------------
CACHED_DIR = str(PROJ_ROOT / "ambulance_dataset_fast_150_espisode_cpu_30_senario" / "cached_llm_v2")
assert os.path.isdir(CACHED_DIR), f"Missing cache dir: {CACHED_DIR}"

# ---------------- Device & seeds ----------------
def _pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        return "mps"
    return "cpu"

def set_global_seeds(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print("Using device:", _pick_device())

# ---------------- Config ----------------
CONFIG = {
    "log_dir": str(PROJ_ROOT / "runs" / "Ego_Emergency"),
    "output_dir": str(PROJ_ROOT),
    "reward": {
        "mode": "custom",
        "custom_cfg": {
            "speed_min": 20.0, "speed_max": 35.0,
            "ahead_dist": 40.0, "same_lane_only": True,
            "ttc_threshold": 3.2, "ttc_w": 1.00,
            "speed_w": 0.90, "clear_bonus": 0.30,
            "block_penalty_w": 0.10, "collision_penalty": 3.0,
            "clip_abs": 1.0,
        }
    },
    "env": {
        "lanes_count": 4, "vehicles_count": 45, "duration": 50,
        "simulation_frequency": 15, "policy_frequency": 1,
        "offscreen_rendering": True, "render_agent": True,
        "show_trajectories": False,
        "high_speed_reward": 0.0, "right_lane_reward": 0.0, "lane_change_reward": 0.0,
        "reward_speed_range": [22, 32], "normalize_reward": False,
        "centering_position": [0.3, 0.5], "scaling": 5.5,
        "scenarios_train": ["highway_emergency_dense", "highway_time_pressure", "highway_merge_heavy", "highway_rush_hour"],
        "scenarios_val":   ["highway_emergency_moderate", "highway_stop_and_go"],
        "scenarios_test":  ["highway_construction", "highway_lane_closure", "highway_accident_scene"],
        "is_emergency": True, "yield_radius": 60.0, "yield_right_bias": 0.7,
        "other_vehicles_type": "rl_langvision.yielding_traffic.YieldingIDM",
    },
    "vision": {"clip_model": "openai/clip-vit-base-patch32", "device": None},
    "text": {"cached_llm_dir": CACHED_DIR, "cached_dim": 384, "local_model": None},
    "policy": {"feat_dim": 512},
    "wrapper": {"clip_stride": 4},
    "algo": {
        "n_envs": 2,
        "ppo": {
            "n_steps": 2048, "batch_size": 512, "n_epochs": 15, "lr": 5e-5,
            "gamma": 0.99, "clip_range": 0.2, "gae_lambda": 0.95,
            "ent_coef": 0.005, "target_kl": 0.04, "vf_coef": 0.5, "max_grad_norm": 0.5,
        },
    },
}

# ---------------- Logging callbacks ----------------
class InfoKeysLogger(BaseCallback):
    def __init__(self, keys=("blockers_ahead","ttc_min","r_speed","r_clear","r_block","r_ttc_pen","r_collision","speed_kph"), verbose=0):
        super().__init__(verbose); self.keys = keys
    def _on_step(self) -> bool:
        infos = self.locals.get("infos") or []
        agg: Dict[str, list] = {k: [] for k in self.keys}
        for d in infos:
            for k in self.keys:
                if k in d: agg[k].append(d[k])
        for k, vals in agg.items():
            if vals:
                try: self.logger.record_mean(f"info/{k}", float(np.mean(vals)))
                except Exception: self.logger.record(f"info/{k}", float(np.mean(vals)))
        return True

class EpisodeRewardComponentsLogger(BaseCallback):
    def __init__(self, keys=("r_speed","r_ttc_pen","r_clear","r_block","r_collision"), verbose=0):
        super().__init__(verbose); self.keys = keys
    def _on_training_start(self) -> None:
        n_envs = self.training_env.num_envs
        self.buf = {k: np.zeros(n_envs, np.float32) for k in self.keys}
        self.crash = np.zeros(n_envs, np.int32)
    def _on_step(self) -> bool:
        infos = self.locals.get("infos") or []
        dones = self.locals.get("dones"); dones_seq = np.asarray(dones).ravel() if dones is not None else ()
        for i, inf in enumerate(infos):
            for k in self.keys:
                if k in inf: self.buf[k][i] += float(inf[k])
            if inf.get("crashed", False) or float(inf.get("r_collision", 0.0)) < 0.0:
                self.crash[i] = 1
            if i < len(dones_seq) and bool(dones_seq[i]):
                for k in self.keys:
                    self.logger.record(f"components/ep_{k}", float(self.buf[k][i])); self.buf[k][i] = 0.0
                self.logger.record("components/ep_collision", float(self.crash[i])); self.crash[i] = 0
        return True

class ConsolePrinter(BaseCallback):
    def __init__(self, max_lines_per_rollout: int = 40, verbose: int = 0):
        super().__init__(verbose); self.max_lines_per_rollout = max_lines_per_rollout
    def _on_training_start(self) -> None:
        n = self.training_env.num_envs
        self.sum_r = np.zeros(n, np.float64); self.len_steps = np.zeros(n, np.int32)
        self.comp_keys = ("r_speed","r_ttc_pen","r_clear","r_block","r_collision")
        self.comp_sum = {k: np.zeros(n, np.float64) for k in self.comp_keys}
        self.blockers_pk = np.zeros(n, np.float64)
        self.speed_sum = np.zeros(n, np.float64); self.speed_cnt = np.zeros(n, np.int32)
        self.crashed = np.zeros(n, np.int32)
        self.finished = []; self.lines_this_rollout = 0; self.rollout_idx = 0
        n_steps = int(self.model.n_steps); rollout = n_steps * n
        device = getattr(self.model, "device", "cpu")
        print(f"=== TRAIN START === device={device} n_envs={n} n_steps={n_steps} rollout_size={rollout}")
    def _on_step(self) -> bool:
        rewards = self.locals.get("rewards"); dones = self.locals.get("dones"); infos = self.locals.get("infos") or []
        if rewards is None: return True
        dones_seq = np.asarray(dones).ravel() if dones is not None else ()
        for i, inf in enumerate(infos):
            self.sum_r[i] += float(rewards[i]); self.len_steps[i] += 1
            for k in self.comp_keys:
                if k in inf: self.comp_sum[k][i] += float(inf[k])
            if "speed_kph" in inf:
                self.speed_sum[i] += float(inf["speed_kph"]); self.speed_cnt[i] += 1
            if "blockers_ahead" in inf:
                self.blockers_pk[i] = max(self.blockers_pk[i], float(inf["blockers_ahead"]))
            if inf.get("crashed", False) or float(inf.get("r_collision", 0.0)) < 0.0:
                self.crashed[i] = 1
            if i < len(dones_seq) and bool(dones_seq[i]):
                spd = (self.speed_sum[i] / max(1, self.speed_cnt[i]))
                row = {"R": self.sum_r[i], "L": int(self.len_steps[i]), "spd": spd,
                       "crash": int(self.crashed[i]),
                       **{k: float(self.comp_sum[k][i]) for k in self.comp_keys},
                       "blk_max": int(self.blockers_pk[i])}
                self.finished.append(row)
                if self.lines_this_rollout < self.max_lines_per_rollout:
                    print(f"[EP] R={row['R']:+.3f}  len={row['L']:4d}  speed={row['spd']:5.1f}kph  "
                          f"blk_max={row['blk_max']:2d}  crash={bool(row['crash'])}  "
                          f"comp:(speed={row['r_speed']:+.3f}, clear={row['r_clear']:+.3f}, "
                          f"block={row['r_block']:+.3f}, ttc_pen=-{row['r_ttc_pen']:.3f}, "
                          f"coll={row['r_collision']:+.3f})")
                    self.lines_this_rollout += 1
                # reset per-env
                self.sum_r[i] = 0.0; self.len_steps[i] = 0
                self.blockers_pk[i] = 0.0; self.crashed[i] = 0
                self.speed_sum[i] = 0.0; self.speed_cnt[i] = 0
                for k in self.comp_keys: self.comp_sum[k][i] = 0.0
        return True
    def _on_rollout_end(self) -> bool:
        self.rollout_idx += 1
        if self.finished:
            R = np.array([e["R"] for e in self.finished], np.float64)
            L = np.array([e["L"] for e in self.finished], np.float64)
            C = np.array([e["crash"] for e in self.finished], np.float64)
            S = np.array([e["spd"] for e in self.finished], np.float64)
            print(f"[ROLLOUT {self.rollout_idx}] episodes={len(self.finished)}  "
                  f"Rmean={R.mean():+.3f}  Lmean={L.mean():.1f}  "
                  f"crash_rate={C.mean():.2f}  speed_mean={S.mean():.1f}kph")
            self.finished.clear()
        self.lines_this_rollout = 0
        return True

# ---------- Best model mirroring (safe) ----------
class BestSaver(BaseCallback):
    """
    Mirror EvalCallback's best zip into:
      - best_model.pkl: model.get_parameters() only (picklable)
      - vecnorm.pkl: current VecNormalize stats
    """
    def __init__(self, train_env: VecNormalize, run_dir: Path, verbose=0):
        super().__init__(verbose)
        self.train_env = train_env
        self.run_dir = Path(run_dir)
        self.best_zip = self.run_dir / "best" / "best_model.zip"   # written by EvalCallback
        self.best_pkl = self.run_dir / "best" / "best_model.pkl"
        self.vecnorm_pkl = self.run_dir / "vecnorm.pkl"
        self._last_best_mtime = None

    def _on_step(self) -> bool:
        if self.best_zip.exists():
            mtime = self.best_zip.stat().st_mtime
            if mtime != self._last_best_mtime:
                self._last_best_mtime = mtime
                # 1) Safe snapshot of parameters only (no TB logger/HMAC issues)
                params = self.model.get_parameters()
                with open(self.best_pkl, "wb") as f:
                    pickle.dump(params, f)
                # 2) Save/update VecNormalize stats
                try:
                    self.train_env.save(str(self.vecnorm_pkl))
                except Exception as e:
                    print(f"[BestSaver] VecNormalize save failed: {e}")
                print(f"💾 Updated: best_model.pkl (params) and vecnorm.pkl")
        return True

# ---------------- Scenario / encoder helpers ----------------
def _choose_scenario(cfg_env: dict, mode: str) -> dict:
    cfg = dict(cfg_env)
    key = {"train":"scenarios_train","eval":"scenarios_val","test":"scenarios_test"}[mode]
    scenarios: Optional[List[str]] = cfg.get(key)
    if scenarios: cfg["scenario"] = random.choice(scenarios)
    return cfg

def _map_clip_name(name: str):
    low = (name or "").lower()
    if "vit-base-patch32" in low: return "ViT-B-32", "openai"
    return name, "openai"

def _make_env_thunk(config: dict, seed: int, mode: str):
    assert mode in {"train","eval","test"}
    def _init():
        cfg = _choose_scenario(config.get("env", {}) or {}, mode)
        base_env = gym.make("highway-v0", render_mode="rgb_array", config=cfg)
        try: base_env.reset(seed=seed)
        except TypeError: pass
        try:
            base_env.action_space.seed(seed)
            base_env.observation_space.seed(seed)
        except Exception: pass

        # Encoders
        vcfg = config.get("vision", {}) or {}
        clip_dev = vcfg.get("device") or _pick_device()
        model_name, pretrained = _map_clip_name(vcfg.get("clip_model", "openai/clip-vit-base-patch32"))
        clip_enc = CLIPImageEncoder(model_name=model_name, pretrained=pretrained, device=clip_dev)

        tcfg = config.get("text", {}) or {}
        if tcfg.get("cached_llm_dir"):
            cached_llm = CachedLLMEmbedder(tcfg["cached_llm_dir"], dim=int(tcfg.get("cached_dim", 384)))
            text_emb = None
        else:
            text_emb = FrozenTextEmbedder(tcfg.get("local_model", "sentence-transformers/all-MiniLM-L6-v2"))
            cached_llm = None

        env = AmbulanceHighwayCLIPWrapper(
            base_env, clip_enc,
            text_embedder=text_emb, cached_llm=cached_llm,
            clip_stride=int(config.get("wrapper", {}).get("clip_stride", 4)),
        )

        # Reward wrapper last
        if (config.get("reward") or {}).get("mode", "custom") == "custom":
            env = SafetySpeedRewardWrapper(env, cfg=config["reward"]["custom_cfg"])

        # Assert wrapper emits dict
        o, _ = env.reset()
        assert isinstance(o, dict) and "clip" in o and "text" in o
        return env
    return _init

def make_vec_envs(config: dict, n_envs: int, base_seed: int, mode: str):
    thunks = [_make_env_thunk(config, base_seed + i, mode) for i in range(max(1, n_envs))]
    venv = DummyVecEnv(thunks)
    venv = VecMonitor(venv)
    # VecNormalize for dict obs
    venv = VecNormalize(venv, norm_obs=True, norm_reward=False, training=(mode=="train"))
    # quick sanity
    o = venv.reset()
    assert isinstance(o, dict) and "clip" in o and "text" in o, "Vec env must emit Dict({'clip','text'})"
    return venv

def _sync_vecnorm_stats(train_env: VecNormalize, other_env: VecNormalize):
    other_env.obs_rms = train_env.obs_rms
    other_env.ret_rms = train_env.ret_rms
    other_env.training = False
    other_env.norm_reward = False

# ---------------- Evaluation ----------------
def _bool_flag(x):
    if isinstance(x, (list, tuple, np.ndarray)): return bool(np.asarray(x).any())
    return bool(x)

def evaluate_model(model: PPO, env: VecNormalize, episodes: int = 10, deterministic: bool = True) -> Dict[str, float]:
    returns, lengths, speed_kph_all = [], [], []
    collisions = 0
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False; truncated = False
        tot = 0.0; steps = 0
        while not (_bool_flag(done) or _bool_flag(truncated)):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, done, truncated, infos = env.step(action)
            tot += float(np.asarray(r).mean()); steps += 1
            info0 = (infos[0] if isinstance(infos, (list, tuple)) and infos else infos) or {}
            if info0.get("crashed", False): collisions += 1
            if "speed_kph" in info0: speed_kph_all.append(float(info0["speed_kph"]))
        returns.append(tot); lengths.append(steps)
    return {
        "return_mean": float(np.mean(returns)) if returns else 0.0,
        "return_std":  float(np.std(returns)) if returns else 0.0,
        "len_mean":    float(np.mean(lengths)) if lengths else 0.0,
        "collisions":  int(collisions),
        "speed_kph_mean": float(np.mean(speed_kph_all)) if speed_kph_all else 0.0,
    }

# ---------------- Training driver ----------------
def print_seed_header(cfg: dict, seed: int, total_timesteps: int):
    a = (cfg.get("algo") or {}).get("ppo", {})
    n_envs = int((cfg.get("algo") or {}).get("n_envs", 1))
    n_steps = int(a.get("n_steps", 2048))
    rollout = n_envs * n_steps
    print("----- RUN CONFIG -----\n"
          f"seed={seed}\n"
          f"timesteps={total_timesteps:,}\n"
          f"n_envs={n_envs}, n_steps={n_steps}, rollout_size={rollout}\n"
          f"lr={a.get('lr')}, batch_size={a.get('batch_size')}, gamma={a.get('gamma')}, "
          f"clip={a.get('clip_range')}, ent_coef={a.get('ent_coef')}, target_kl={a.get('target_kl')}\n"
          f"reward_mode={(cfg.get('reward') or {}).get('mode', 'custom')}\n"
          f"train_scenarios={cfg['env'].get('scenarios_train')}\n"
          f"val_scenarios={cfg['env'].get('scenarios_val')}\n"
          f"test_scenarios={cfg['env'].get('scenarios_test')}\n"
          "----------------------")

def train_one_seed(config: dict, seed: int, total_timesteps: int, model_prefix: str):
    set_global_seeds(seed)
    n_envs = int((config.get("algo") or {}).get("n_envs", 1))

    tb_root = Path(config.get("log_dir") or (PROJ_ROOT / "runs" / "ppo_simple"))
    run_dir = tb_root / f"seed_{seed}"
    ckpt_dir = run_dir / "ckpt"
    best_dir = run_dir / "best"
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_dir.mkdir(parents=True, exist_ok=True)

    # build envs
    env = make_vec_envs(config, n_envs, base_seed=seed, mode="train")
    eval_env = make_vec_envs(config, 1, base_seed=seed + 10_000, mode="eval")
    _sync_vecnorm_stats(env, eval_env)

    # policy
    policy_kwargs = dict(
        features_extractor_class=CLIPLangExtractor,
        features_extractor_kwargs=dict(features_dim=(config.get("policy") or {}).get("feat_dim", 512)),
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    )
    a = (config.get("algo") or {}).get("ppo", {}) or {}
    model = PPO(
        "MultiInputPolicy", env, seed=seed,
        policy_kwargs=policy_kwargs,
        n_steps=a.get("n_steps", 2048),
        batch_size=a.get("batch_size", 512),
        n_epochs=a.get("n_epochs", 15),
        learning_rate=a.get("lr", 5e-5),
        gamma=a.get("gamma", 0.99),
        clip_range=a.get("clip_range", 0.2),
        gae_lambda=a.get("gae_lambda", 0.95),
        ent_coef=a.get("ent_coef", 0.005),
        target_kl=a.get("target_kl", 0.04),
        verbose=1, tensorboard_log=str(tb_root), device=_pick_device(),
    )

    # callbacks
    console_cb = ConsolePrinter(max_lines_per_rollout=40)
    info_cb = InfoKeysLogger()
    comp_cb = EpisodeRewardComponentsLogger()
    eval_cb = EvalCallback(
        eval_env, best_model_save_path=str(best_dir),
        log_path=str(run_dir / "eval"),
        eval_freq=max(1, (10_000 // max(1, n_envs))),
        n_eval_episodes=20, deterministic=True, render=False,
    )
    best_mirror_cb = BestSaver(train_env=env, run_dir=run_dir)
    ckpt_cb = CheckpointCallback(
        save_freq=max(1, (50_000 // max(1, n_envs))),
        save_path=str(ckpt_dir), name_prefix="ppo"
    )

    print_seed_header(config, seed, total_timesteps)
    print(f"[seed {seed}] learn() for {total_timesteps:,} steps on {model.device} …")
    t0 = time.time()
    model.learn(
        total_timesteps=total_timesteps,
        tb_log_name=f"seed_{seed}",
        callback=[console_cb, info_cb, comp_cb, eval_cb, best_mirror_cb, ckpt_cb],
    )
    print(f"[seed {seed}] learn() done in {(time.time()-t0)/60:.1f} min")

    # final saves
    model_path = Path(config.get("output_dir", PROJ_ROOT)) / f"{model_prefix}_seed{seed}.zip"
    model.save(str(model_path))
    # also save a final vecnorm snapshot
    env.save(str(run_dir / "vecnorm.pkl"))
    print(f"[seed {seed}] saved -> {model_path}")
    print(f"[seed {seed}] best model dir -> {best_dir}")
    print(f"[seed {seed}] vecnorm -> {run_dir / 'vecnorm.pkl'}")

    # quick metrics
    val_metrics = evaluate_model(model, eval_env, episodes=20, deterministic=True)
    (run_dir / "val_metrics.json").write_text(json.dumps(val_metrics, indent=2))
    print(f"[seed {seed}] VAL:", val_metrics)

    test_env = make_vec_envs(config, 1, base_seed=seed + 20_000, mode="test")
    _sync_vecnorm_stats(env, test_env)
    test_metrics = evaluate_model(model, test_env, episodes=20, deterministic=True)
    (run_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
    print(f"[seed {seed}] TEST:", test_metrics)

    env.close(); eval_env.close(); test_env.close()

def train(config: dict, seeds: Iterable[int] | None = None,
          total_timesteps: int = 100_000, model_prefix: str = "ppo_clip_vlm_f2"):
    seeds = list(seeds) if seeds is not None else [0]
    for s in seeds:
        train_one_seed(config, seed=s, total_timesteps=total_timesteps, model_prefix=model_prefix)

# ---------------- Main ----------------
if __name__ == "__main__":
    # 1) cache sanity
    embed_info = os.path.join(CACHED_DIR, "embed_info.json")
    assert os.path.exists(embed_info)
    _info = json.loads(open(embed_info).read())
    assert int(_info["dim"]) == 384
    parqs = glob.glob(os.path.join(CACHED_DIR, "*.parquet")); assert parqs
    _df = pd.read_parquet(parqs[0])
    for col in ["episode_id","step","embedding","text"]: assert col in _df.columns
    assert len(_df["embedding"].iloc[0]) == 384
    print("[ok] cache sanity:", Path(parqs[0]).name, "rows=", len(_df))

    # 2) smoke env (VecNormalize wraps dict => reset returns dict)
    _env = make_vec_envs(CONFIG, 1, base_seed=0, mode="train")
    obs = _env.reset()
    assert isinstance(obs, dict) and all(k in obs for k in ("clip","text")), "Obs must have 'clip' and 'text'"
    print("Obs keys:", list(obs.keys()))
    _env.close()

    # 3) train
    set_global_seeds(0)
    train(CONFIG, seeds=[0, 1], total_timesteps=75_000, model_prefix="ppo_clip_vlm_f2")


# Evaluate

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# rebuild eval env exactly like training
eval_env = make_vec_envs(CONFIG, 1, base_seed=0, mode="eval")
eval_env = VecNormalize.load(".../seed_0/best/vecnorm.pkl", eval_env)
eval_env.training = False
eval_env.norm_reward = False

model = PPO.load(".../seed_0/best/best_model.zip", device="cuda")
